## 08 POI Demand

**笔记本**：`08 poi_demand20260403.1.ipynb`

**库**：`geopandas`、`pandas`、`numpy`、`matplotlib`、`shapely`（`Point`、`prepared`）等。

**输入**：
- `../04 Transport/data_raw/shenzhen_boundary.geojson`
- `../00/02-poi-aoi/1-poi-2026.01/SHP/*.shp`（多类 POI shapefile，映射为 food/retail/medical 等 demand 大类）
- （格网）优先 `../06 Buildings/data_out/sz_building_grid.gpkg`，否则 `../03 Boundary/data_out/sz_hex_grid_res8.gpkg`，再否则笔记本内 `h3.geo_to_cells`

**做了什么 / 算了什么**：
- 批量读取本地 POI，赋予 `poi_type` / `demand_group`；按名称+坐标 **去重**；裁剪深圳；统计各类数量。
- **H3 res=8**（`h3_id`）：优先与 06 一致，否则与 03b 导出一致；空间连接统计每格 POI 计数与 **demand_pressure**（等综合强度）。

**写出文件**：
- `data_out/sz_poi.gpkg`
- `data_out/sz_demand_grid.gpkg`

**典型输出信息**：各类 POI 条数、去重/裁剪后总量、网格数、有 POI 的格数、`demand_pressure` 范围与保存路径。
- 不同类型 POI 对“配送/活动需求”的贡献不一样；demand_pressure做proxy，用 POI 结构近似表示潜在需求强度。
- 标识高需求区的格子；和 friction 叠加，看“补位价值”；支撑 observed vs random 分析（现有站点是不是更偏向高需求空间）；支撑新增选址模拟
- 注意：POI 数量不等于真实订单量；权重是分析设定，不是客观真理）（food 0.25、retail 0.20 是建模选择。）；POI 更偏“静态设施分布”，它反映的是空间上的设施密度， 不是时段化、实时化的配送压力。潜在需求压力 proxy，而不是“真实即时需求”。）



In [1]:
# ============================================================
#  08 POI Demand Layers
#  数据源: 00/ 本地 POI 数据 (2026.01, ~490K 条, 无需 API)
#  输出: POI 点位 + 网格化需求密度
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely.prepared import prep as shapely_prep
import warnings
warnings.filterwarnings("ignore")

OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

GRID_06 = Path("../06 Buildings/data_out/sz_building_grid.gpkg")
GRID_03_R8 = Path("../03 Boundary/data_out/sz_hex_grid_res8.gpkg")

BOUNDARY = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")

# 00 本地 POI 文件 → demand 分类
POI_BASE = Path("../00/02-poi-aoi/1-poi-2026.01/SHP")
POI_FILES = {
    "shenzhen_can_yin_fu_wu.shp":     "food",
    "shenzhen_gou_wu_fu_wu.shp":     "retail",
    "shenzhen_yi_liao_bao_jian_fu_wu.shp":  "medical",
    "shenzhen_ke_jiao_wen_hua_fu_wu.shp":  "education",
    "shenzhen_shang_wu_zhu_zhai.shp":     "office",
    "shenzhen_ti_yu_xiu_xian_fu_wu.shp":  "leisure",
    "shenzhen_feng_jing_ming_sheng.shp":     "scenic",
    "shenzhen_sheng_huo_fu_wu.shp":     "service",
}

shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union
sz_prep = shapely_prep(shenzhen_geom)

print(f"POI source: {POI_BASE}")
print(f"Categories: {len(POI_FILES)}")

POI source: ../00/02-poi-aoi/1-poi-2026.01/SHP
Categories: 8


In [2]:
# ============================================================
#  1. 读取本地 POI 数据 (00/2026.01, 秒级完成)
# ============================================================

all_frames = []
for filename, demand_group in POI_FILES.items():
    path = POI_BASE / filename
    if not path.exists():
        print(f"  MISSING: {path}")
        continue
    gdf = gpd.read_file(path)
    gdf["poi_type"] = demand_group
    keep = ["name", "address", "poi_type", "geometry"]
    keep = [c for c in keep if c in gdf.columns]
    all_frames.append(gdf[keep])
    print(f"  {demand_group:12s}  {len(gdf):>9,}  <- {filename}")

poi_raw = pd.concat(all_frames, ignore_index=True)
print(f"\nTotal raw POIs: {len(poi_raw):,}")

  food            108,953  <- shenzhen_can_yin_fu_wu.shp
  retail          162,881  <- shenzhen_gou_wu_fu_wu.shp
  medical          26,369  <- shenzhen_yi_liao_bao_jian_fu_wu.shp
  education        24,709  <- shenzhen_ke_jiao_wen_hua_fu_wu.shp
  office           35,892  <- shenzhen_shang_wu_zhu_zhai.shp
  leisure          24,409  <- shenzhen_ti_yu_xiu_xian_fu_wu.shp
  scenic            5,875  <- shenzhen_feng_jing_ming_sheng.shp
  service         101,544  <- shenzhen_sheng_huo_fu_wu.shp

Total raw POIs: 490,632


In [3]:
# ============================================================
#  2. 清洗去重 + 裁剪
# ============================================================

poi_gdf = gpd.GeoDataFrame(poi_raw, crs=4326)
print(f"Raw: {len(poi_gdf):,}")

# 去重: 相同名称 + 相近坐标
poi_gdf["_lon"] = poi_gdf.geometry.x.round(5)
poi_gdf["_lat"] = poi_gdf.geometry.y.round(5)
poi_gdf = poi_gdf.drop_duplicates(subset=["name", "_lon", "_lat"]).drop(columns=["_lon", "_lat"])
print(f"After dedup: {len(poi_gdf):,}")

# 裁剪到深圳边界
poi_gdf = gpd.clip(poi_gdf, shenzhen_geom)
print(f"After clip: {len(poi_gdf):,}")

# 添加 lon/lat 列 (供下游使用)
poi_gdf["lon"] = poi_gdf.geometry.x
poi_gdf["lat"] = poi_gdf.geometry.y

# demand_group = poi_type 本身 (已经是大类)
poi_gdf["demand_group"] = poi_gdf["poi_type"]

print(f"\n=== POI by type ===")
print(poi_gdf["poi_type"].value_counts().to_string())

Raw: 490,632
After dedup: 478,437
After clip: 478,246

=== POI by type ===
poi_type
retail       156754
food         107650
service       99690
office        34841
medical       26044
education     23966
leisure       23534
scenic         5767


In [4]:
# ============================================================
#  3. 网格化需求密度 + demand_pressure
#  H3 res=8：优先 06 → 其次 03b sz_hex_grid_res8 → 最后 geo_to_cells
# ============================================================

if GRID_06.exists():
    g6 = gpd.read_file(GRID_06)
    grid = g6[["h3_id", "geometry"]].copy()
    grid["h3_id"] = grid["h3_id"].astype(str)
    print(f"Grid from 06: {len(grid):,} H3 cells")
elif GRID_03_R8.exists():
    g3 = gpd.read_file(GRID_03_R8)
    grid = g3[["h3_id", "geometry"]].copy()
    grid["h3_id"] = grid["h3_id"].astype(str)
    print(f"Grid from 03b: {len(grid):,} H3 cells ({GRID_03_R8.name})")
else:
    import h3
    from shapely.geometry import Polygon, mapping

    def _h3_to_polygon(h: str) -> Polygon:
        coords = h3.cell_to_boundary(h)
        return Polygon([(lng, lat) for lat, lng in coords])

    H3_RES = 8
    poly_gj = mapping(shenzhen_geom)
    hex_ids = sorted(h3.geo_to_cells(poly_gj, res=H3_RES))
    grid = gpd.GeoDataFrame(
        {"h3_id": hex_ids},
        geometry=[_h3_to_polygon(h) for h in hex_ids],
        crs=4326,
    )
    print(f"Grid (fallback H3): {len(grid):,} cells — 建议先运行 03b 与 06")

# 空间连接 POI → 网格
joined = gpd.sjoin(poi_gdf, grid[["h3_id", "geometry"]], how="inner", predicate="within")

# 按需求大类分组统计
pivot = joined.groupby(["h3_id", "demand_group"]).size().unstack(fill_value=0)
pivot.columns = [f"{c}_count" for c in pivot.columns]
pivot["poi_total"] = pivot.sum(axis=1)
pivot = pivot.reset_index()

# 合并回网格
grid_demand = grid.merge(pivot, on="h3_id", how="left").fillna(0)

# demand_pressure: 加权综合需求指数
WEIGHTS = {
    "food_count": 0.25,
    "retail_count": 0.20,
    "medical_count": 0.15,
    "office_count": 0.15,
    "education_count": 0.05,
    "service_count": 0.05,
    "leisure_count": 0.05,
    "scenic_count": 0.05,
    "residential_count": 0.05,
}

grid_demand["demand_pressure"] = 0
for col, w in WEIGHTS.items():
    if col in grid_demand.columns:
        grid_demand["demand_pressure"] += w * grid_demand[col]

# 归一化到 [0, 1]
dp = grid_demand["demand_pressure"]
dp_max = dp.max()
if dp_max > 0:
    grid_demand["demand_pressure_norm"] = dp / dp_max

non_empty = (grid_demand["poi_total"] > 0).sum()
print(f"Grids with POIs: {non_empty:,} / {len(grid_demand):,}")
print(f"Demand pressure: max={dp.max():.1f}, mean(non-empty)={dp[dp > 0].mean():.1f}")


Grid from 06: 2,754 H3 cells
Grids with POIs: 2,125 / 2,754
Demand pressure: max=416.0, mean(non-empty)=35.2


In [5]:
# ============================================================
#  4. 保存 + 统计摘要
# ============================================================

# POI 点位
poi_out = poi_gdf[["name", "poi_type", "demand_group", "address", "lon", "lat", "geometry"]]
poi_out.to_file(OUT / "sz_poi.gpkg", driver="GPKG")
print(f"POI      -> data_out/sz_poi.gpkg          ({len(poi_out):,} rows)")

# 需求网格
grid_demand.to_file(OUT / "sz_demand_grid.gpkg", driver="GPKG")
print(f"Grid     -> data_out/sz_demand_grid.gpkg   ({len(grid_demand):,} cells)")

# 统计摘要
print("\n=== POI Summary ===")
print(f"Total POIs: {len(poi_out):,}")
print(poi_out["poi_type"].value_counts().to_string())

print("\n=== Demand Grid Summary (non-empty) ===")
g = grid_demand[grid_demand["poi_total"] > 0]
for col in [c for c in grid_demand.columns if c.endswith("_count")] + ["demand_pressure"]:
    if col in g.columns:
        print(f"  {col:25s}  mean={g[col].mean():8.1f}  max={g[col].max():8.0f}")

POI      -> data_out/sz_poi.gpkg          (478,246 rows)
Grid     -> data_out/sz_demand_grid.gpkg   (2,754 cells)

=== POI Summary ===
Total POIs: 478,246
poi_type
retail       156754
food         107650
service       99690
office        34841
medical       26044
education     23966
leisure       23534
scenic         5767

=== Demand Grid Summary (non-empty) ===
  education_count            mean=    11.2  max=     121
  food_count                 mean=    50.5  max=     550
  leisure_count              mean=    11.0  max=     165
  medical_count              mean=    12.2  max=     175
  office_count               mean=    16.3  max=     132
  retail_count               mean=    73.6  max=    1588
  scenic_count               mean=     2.7  max=     187
  service_count              mean=    46.8  max=     583
  demand_pressure            mean=    35.2  max=     416
